# Autoregressive rollout vs direct — straight-path weld temperature

For **FNO**, **RIGNO**, and **DeepONet** we trained two models on the same
straight-path varying-(P,V) dataset (65/9/9 train/val/test cases, 20 frames each,
grid 90×40×30, lab frame):

- **direct** (`*_tb`): one-step model. Every frame is predicted from the
  *ground-truth* previous state — teacher forcing. At eval it scores each frame
  independently against GT.
- **rollout** (`*_ar`): autoregressive model. Seeded with GT at t=0, then its own
  prediction is fed back as the previous-state channels (`T_prev`, `alpha_prev`, …)
  for the next frame. It scores frames 1..19.

This notebook loads both evals for the 9-case test split, reproduces the
Temperature-MAE table, plots per-frame error growth, and shows GT / direct /
rollout side by side (top = fields, right = error maps) so the rollout failure
modes are visible where volume metrics hide them.

All temperatures are normalized by `ref_scales.T = 3000 K`; multiply by 3000 for
Kelvin.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

REPO = Path.cwd().resolve().parents[1]  # notebook lives in src/notebooks/
TEST_DIR = REPO / "data/flow3D/resampled_straight_path_data/test"
T_FRAMES = 20          # frames per case
REF_T = 3000.0         # ref_scales.T -> Kelvin

# tb = direct one-step (teacher-forced); ar = autoregressive rollout.
# RIGNO's first AR run (rigno_straight_ar) froze at step 500 in the WarmupNode
# alpha-mask trap; ar2 is the masking-off retrain that actually learned.
EVAL = {
    "FNO":      {"direct": "fno_straight_tb_test_0706_1843",
                 "rollout": "fno_straight_ar_test_0706_1937"},
    "RIGNO":    {"direct": "rigno_straight_tb_test_0706_2234",
                 "rollout": "rigno_straight_ar2_test_0707_0234"},
    "DeepONet": {"direct": "dpn_straight_tb_test_0706_2335",
                 "rollout": "dpn_straight_ar_test_0707_0211"},
}

In [ ]:
def load_direct(run, field="T"):
    # direct eval writes flat (B*T, C, X, Y, Z), case-major -> (cases, T, C, X, Y, Z)
    ev = REPO / "outputs" / run / "fno_eval"
    p = np.load(ev / "predictions.npz")[field].astype("f8")
    g = np.load(ev / "ground_truth.npz")[field].astype("f8")
    c = p.shape[0] // T_FRAMES
    resh = lambda a: a.reshape(c, T_FRAMES, *a.shape[1:]) * REF_T
    return resh(p), resh(g)                      # (9, 20, 1, 90, 40, 30) in K


def load_rollout(run, field="T"):
    # rollout eval writes (cases, T-1, C, X, Y, Z) already, frames 1..19
    ev = REPO / "outputs" / run / "fno_eval"
    p = np.load(ev / "rollout_predictions.npz")[field].astype("f8") * REF_T
    g = np.load(ev / "ground_truth_rollout.npz")[field].astype("f8") * REF_T
    return p, g                                  # (9, 19, 1, 90, 40, 30) in K


DATA = {}
for fam, runs in EVAL.items():
    dp, dg = load_direct(runs["direct"])
    rp, rg = load_rollout(runs["rollout"])
    DATA[fam] = {"direct": dp, "direct_gt": dg, "rollout": rp, "rollout_gt": rg}
    print(f"{fam:9s}  direct {dp.shape}   rollout {rp.shape}")

# coords + case order (eval iterates cases in sorted filename order)
files = sorted(f for f in TEST_DIR.glob("*.npz") if not f.name.startswith("var_"))
d0 = np.load(files[0])
X, Y, Z = d0["x"], d0["y"], d0["z"]
Z0 = int(np.argmin(np.abs(Z)))                   # surface z=0 index
NAMES = [f.stem.replace("_line_track_vtk", "") for f in files]
print(f"grid {X.size}x{Y.size}x{Z.size}; surface z=0 at index {Z0}")
print("test cases:", NAMES)

# Alignment: rollout frame j is physical frame j+1, i.e. rollout[:, t-1] <-> direct[:, t].
# The direct eval GT applies the boiling-fill alpha mask (T=3123 K at keyhole/void
# cells); the rollout eval GT does not (~0.5% of cells, mean ~8 K). Each method is
# therefore scored against its OWN GT below, matching the reported numbers.
_off = np.abs(DATA["RIGNO"]["direct_gt"][:, 1:] - DATA["RIGNO"]["rollout_gt"]).mean()
print(f"direct-vs-rollout GT convention gap (masking): {_off:.2f} K mean")

## Temperature MAE: rollout vs direct

Recomputed from the arrays over frames 1..19 (drop t=0 so the window matches the
rollout).

In [ ]:
print(f"{'family':9} {'direct t1..19':>16} {'rollout t1..19':>16} {'ratio':>7}")
for fam, d in DATA.items():
    dir_e = np.abs(d["direct"][:, 1:] - d["direct_gt"][:, 1:]).mean()   # K, frames 1..19
    rol_e = np.abs(d["rollout"] - d["rollout_gt"]).mean()              # K, frames 1..19
    print(f"{fam:9} {dir_e:9.1f} K     {rol_e:9.1f} K     {rol_e / dir_e:6.2f}x")

## Per-frame error growth

Where the rollout diverges from direct. The direct curve (dashed) is flat — each
frame is an independent one-step prediction. The rollout curve (solid) is the
accumulation: FNO stays flat, RIGNO grows ~linearly, DeepONet blows up. Note the
log y-axis. Right panel: rollout error concentrates just below the z=0 surface
(the gas slices above are ~zero and dilute the volume average).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
frames = np.arange(1, T_FRAMES)
for fam, d in DATA.items():
    rol = np.abs(d["rollout"] - d["rollout_gt"]).mean(axis=(0, 2, 3, 4, 5))       # (19,) K
    dirf = np.abs(d["direct"][:, 1:] - d["direct_gt"][:, 1:]).mean(axis=(0, 2, 3, 4, 5))
    line, = ax[0].plot(frames, rol, "-o", ms=3, label=f"{fam} rollout")
    ax[0].plot(frames, dirf, "--", color=line.get_color(), alpha=0.5, label=f"{fam} direct")
ax[0].set_yscale("log")
ax[0].set_xlabel("frame index"); ax[0].set_ylabel("T MAE [K]")
ax[0].set_title("per-frame T MAE — rollout (solid) vs direct (dashed)")
ax[0].legend(fontsize=7)

for fam, d in DATA.items():
    e = np.abs(d["rollout"] - d["rollout_gt"]).mean(axis=(0, 1, 2, 3, 4))          # (Z,) K
    ax[1].plot(Z * 1e6, e, label=fam)
ax[1].axvline(0.0, color="k", lw=0.6, ls="--")
ax[1].set_xlabel("z [um]  (dashed: z=0 surface)"); ax[1].set_ylabel("rollout T MAE [K]")
ax[1].set_title("rollout error vs depth")
ax[1].legend(fontsize=8)
plt.show()

## GT vs direct vs rollout — surface maps

Surface-normal (x–y) slice at z=0 for one test case, at four frames. Columns:
ground truth, direct prediction, rollout prediction, then the two error maps
(shared symmetric scale). The cyan line is the weld centerline y=0. Direct tracks
GT almost exactly; the rollout column is where the accumulated error lives.

In [ ]:
def compare_static(fam, case=0, z_idx=None, frames=(1, 6, 12, 19)):
    z_idx = Z0 if z_idx is None else z_idx
    d = DATA[fam]
    ext = [X.min() * 1e3, X.max() * 1e3, Y.min() * 1e3, Y.max() * 1e3]   # mm
    fig, ax = plt.subplots(len(frames), 5, figsize=(15, 2.7 * len(frames)),
                           constrained_layout=True)
    for i, t in enumerate(frames):
        gt = d["direct_gt"][case, t, 0, :, :, z_idx]
        di = d["direct"][case, t, 0, :, :, z_idx]
        ro = d["rollout"][case, t - 1, 0, :, :, z_idx]
        rgt = d["rollout_gt"][case, t - 1, 0, :, :, z_idx]
        vmax = float(max(gt.max(), di.max(), ro.max()))
        ec = float(max(50.0, np.abs(di - gt).max(), np.abs(ro - rgt).max()))
        panels = [
            (gt, "GT", "inferno", 280, vmax),
            (di, "direct", "inferno", 280, vmax),
            (ro, "rollout", "inferno", 280, vmax),
            (di - gt, "direct err", "coolwarm", -ec, ec),
            (ro - rgt, "rollout err", "coolwarm", -ec, ec),
        ]
        for j, (arr, ttl, cmap, lo, hi) in enumerate(panels):
            im = ax[i, j].imshow(arr.T, origin="lower", extent=ext, cmap=cmap, aspect="auto")
            im.set_clim(lo, hi)
            ax[i, j].axhline(0.0, color="cyan", lw=0.5, alpha=0.5)
            ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
            if i == 0:
                ax[i, j].set_title(ttl, fontsize=10)
            if j == 0:
                ax[i, j].set_ylabel(f"frame {t}", fontsize=9)
        fig.colorbar(im, ax=ax[i, -1], shrink=0.8)
    fig.suptitle(f"{fam}  |  case {NAMES[case]}  |  surface z=0", fontsize=12)
    plt.show()


for fam in DATA:
    compare_static(fam)

## Interactive viewer

Pick a family, case, frame (1..19), and depth. Top row GT / direct / rollout,
bottom row the two error maps.

In [ ]:
def view(fam, case, t, z_idx):
    d = DATA[fam]
    gt = d["direct_gt"][case, t, 0, :, :, z_idx]
    di = d["direct"][case, t, 0, :, :, z_idx]
    ro = d["rollout"][case, t - 1, 0, :, :, z_idx]
    rgt = d["rollout_gt"][case, t - 1, 0, :, :, z_idx]
    ext = [X.min() * 1e3, X.max() * 1e3, Y.min() * 1e3, Y.max() * 1e3]
    vmax = float(max(gt.max(), di.max(), ro.max()))
    ec = float(max(50.0, np.abs(di - gt).max(), np.abs(ro - rgt).max()))
    fig, ax = plt.subplots(2, 3, figsize=(12, 6.5), constrained_layout=True)
    top = [(gt, "GT"), (di, "direct"), (ro, "rollout")]
    bot = [(None, ""), (di - gt, "direct err"), (ro - rgt, "rollout err")]
    for a, (arr, ttl) in zip(ax[0], top):
        im = a.imshow(arr.T, origin="lower", extent=ext, cmap="inferno", aspect="auto")
        im.set_clim(280, vmax); a.set_title(ttl); a.axhline(0, color="cyan", lw=0.6, alpha=0.6)
        fig.colorbar(im, ax=a, shrink=0.8)
    for a, (arr, ttl) in zip(ax[1], bot):
        if arr is None:
            a.axis("off"); continue
        im = a.imshow(arr.T, origin="lower", extent=ext, cmap="coolwarm", aspect="auto")
        im.set_clim(-ec, ec); a.set_title(f"{ttl} (MAE {np.abs(arr).mean():.0f} K)")
        a.axhline(0, color="k", lw=0.6, alpha=0.4); fig.colorbar(im, ax=a, shrink=0.8)
    fig.suptitle(f"{fam}  |  {NAMES[case]}  |  frame {t}  |  z={Z[z_idx] * 1e6:.0f} um", fontsize=11)
    plt.show()


widgets.interact(
    view,
    fam=widgets.Dropdown(options=list(DATA), description="model"),
    case=widgets.IntSlider(min=0, max=len(NAMES) - 1, value=0, description="case"),
    t=widgets.IntSlider(min=1, max=T_FRAMES - 1, value=T_FRAMES - 1, description="frame"),
    z_idx=widgets.IntSlider(min=0, max=Z.size - 1, value=Z0, description="z"),
);

## Why the rollout degrades — and how to mitigate it

### Why direct is easy and rollout is hard

Both models are trained identically: **one-step, teacher-forced**. Each training
pair is (GT context at t+1, GT state at t) → GT state at t+1
(`preprocessing_datasets.py:690-699`), and the loss is a plain single-step
supervised constraint with no unrolling (`model_domain.py:267-292`). The feedback
channels are `T_prev` and `alpha_prev` (the two `state_keys`).

- **Direct eval** keeps feeding GT `*_prev`, so the model runs on exactly the
  input distribution it trained on. Every frame is an independent one-step
  prediction — the per-frame curve is flat (dashed lines above).
- **Rollout eval** seeds GT only at t=0, then feeds the model's *own* prediction
  back as `*_prev` (`postprocessing.py:448`). The model now runs on inputs it
  never saw in training: its own imperfect, off-distribution state. That is
  **exposure bias** — the classic autoregressive failure. Each step's error
  becomes the next step's input error, and whether it grows or decays is set by
  the model's input→output gain.

### Why FNO holds and RIGNO / DeepONet don't — spatial inductive bias

The three architectures damp fed-back error very differently:

| model | mechanism | spatial coupling | rollout behaviour |
|---|---|---|---|
| **FNO** | global spectral conv (`SpectralConv`) | whole grid, high freqs truncated | gain < 1 → **flat** (1.3×) |
| **RIGNO** | graph message passing (`scatter_mean` over neighbours, `rigno_arch.py:220`) | local, a few hops | gain ≈ 1 → **linear** (3.2×) |
| **DeepONet** | pointwise branch·trunk dot | none — each query point is independent | gain > 1 → **blow-up** (24×) |

FNO's spectral truncation acts as a low-pass filter on the fed-back state, so
injected high-frequency error decays. RIGNO's message passing smooths locally but
its regional-graph decode also *injects* structured artifacts — visible in the
maps as the cold hole at the pool centre and the concentric rings that grow with
frame. DeepONet has **no spatial coupling** (each output point is an independent
branch·trunk product), so nothing damps the high-frequency feedback noise and it
compounds fastest.

Two amplifiers specific to this eval:

- **No physical clamp on the fed-back state.** `_apply_prediction_postprocessing`
  (`postprocessing.py:262-308`) re-derives `fl` bounded and optionally
  cosmetic-masks `T`, but never clips `alpha` to [0,1] or floors `T` at ambient.
  The DeepONet rollout drifts to `alpha ≈ 1.28` and `T ≈ −1500 K` — unphysical
  states that then feed back and compound.
- **DeepONet conditioning.** `T_prev`/`alpha_prev` are spatially-varying fields;
  they only reach the trunk because the AR run lists them in
  `deeponet_trunk_extra_keys` — otherwise they fall into the branch, which
  assumes spatially-constant conditioning (`config_loading.py:782-785`). Even
  routed correctly, the low-rank branch·trunk basis reconstructs a fed-back field
  imperfectly, and that reconstruction error is what compounds.

### Mitigations (ranked by effort)

None of these are needed if FNO is acceptable — on this dataset FNO has the best
direct *and* rollout MAE, so **the laziest fix is to use FNO for rollout.** The
rest matter when you specifically want RIGNO's or DeepONet's inductive bias (e.g.
RIGNO beats FNO one-step on the random-path data).

1. **Physical clamp of the fed-back state (no retrain).** Add
   `alpha = np.clip(alpha, 0, 1)` and a `T ≥ ambient` floor at the single
   write-back choke point (`postprocessing.py:445`, just before `:448`). Stops
   the unphysical DeepONet drift for free and is testable on the existing
   checkpoints.
2. **Pushforward / noise-injection training (cheap retrain).** The hook already
   exists: `autoregressive.prev_noise_std > 0` perturbs the GT `*_prev` at train
   time (`preprocessing_datasets.py:691-694`), teaching the model to correct its
   own error distribution instead of assuming clean inputs. It was 0 in these
   runs. Improvement: resample the noise per batch (it is currently static per
   epoch) and scale it to the model's own one-step error. This is the principled
   exposure-bias fix and usually the best return per GPU-hour.
3. **Multi-step unrolled training loss (bigger change).** Unroll k steps during
   training and backprop through the rollout so the loss sees compounding
   directly. Not implemented today; it would replace the single-step constraint
   at `model_domain.py:267-292`, reusing the inference unroll
   (`postprocessing.py:416-448`) as the differentiable template.

### Hybrid ideas

- **FNO-anchored rollout (the strongest hybrid).** FNO's rollout is stable, so use
  its predicted `T`/`alpha` field as the `*_prev` conditioning fed to
  RIGNO/DeepONet, and let them add the sharp keyhole/pool detail as a residual on
  top. This grafts FNO's rollout stability onto the sharper one-step models.
  Code-wise it is a small change to the rollout loop: source `step_input[*_prev]`
  from the FNO rollout instead of the model's own output
  (`postprocessing.py:419`).
- **Smooth the fed-back state (especially DeepONet).** Apply a light
  Gaussian/diffusion filter to `*_prev` before feed-back — a cheap inference-only
  way to inject the spatial coupling DeepONet structurally lacks, at the same
  choke point as the clamp.
- **Consensus / re-anchoring.** Average an ensemble's fed-back state, or (if any
  future GT frames are available) periodically re-seed with GT to reset the drift.